# Pipeline Observability

The `neo4j_graphrag.pipeline` DSL is pure data: a `Pipeline` is a graph of `Operator` nodes,
and an *interpreter* evaluates it. `LocalInterpreter` accepts a list of **`StageObserver`**
instances and wraps every operator's output stream with them — no changes to the pipeline
definition required.

Each observer gets three hooks, fired **per item, per stage boundary**:

| hook | when |
| --- | --- |
| `before(op, item)` | `op` has produced `item`, just before it flows downstream |
| `after(op, item)` | `item` has been consumed by every downstream stage |
| `on_error(op, error)` | evaluating `op` raised; the exception propagates afterwards |

Everything stays lazy: hooks only fire for items that are actually pulled from the stream.

In [1]:
import logging
import time
from collections import defaultdict

from neo4j_graphrag.pipeline import (
    LocalInterpreter,
    LoggingStageObserver,
    Pipeline,
    StageObserver,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## 1. Watching a pipeline run: `LoggingStageObserver`

The built-in `LoggingStageObserver` logs stage start/finish (with item counts) at `INFO`,
each item at `DEBUG`, and failures at `ERROR`.

Note how laziness shows up in the logs: `take(3)` stops pulling once it has 3 items, so the
upstream `Filter`, `Map`, and `SourceOp` stages are abandoned mid-stream and never log
"finished" — while `Take` drains its own stream and does.

In [2]:
pipeline = (
    Pipeline(range(20))
    .map(lambda x: x * x)
    .filter(lambda x: x % 2 == 0)
    .take(3)
)

result = list(LocalInterpreter(observers=[LoggingStageObserver()]).evaluate(pipeline))
result

INFO neo4j_graphrag.pipeline.observers: Stage Take: starting


INFO neo4j_graphrag.pipeline.observers: Stage Filter: starting


INFO neo4j_graphrag.pipeline.observers: Stage Map: starting


INFO neo4j_graphrag.pipeline.observers: Stage SourceOp: starting


INFO neo4j_graphrag.pipeline.observers: Stage Take: finished, 3 item(s)


[0, 4, 16]

## 2. Custom observer: per-stage throughput

`StageObserver`'s hooks are no-ops by default — override only what you need. Here we count
items per stage and measure the wall-clock window each stage was active, which gives a rough
throughput figure per stage boundary.

One caveat for timing in lazy pipelines: stages overlap (an item can be partway through
several stages at once), so per-stage wall times are windows of activity, not exclusive
compute cost.

In [3]:
class TimingObserver(StageObserver):
    """Counts items per stage and records each stage's active time window."""

    def __init__(self):
        self.items = defaultdict(int)
        self._first_seen = {}
        self._last_seen = {}

    def before(self, op, item):
        now = time.perf_counter()
        self.items[op.name] += 1
        self._first_seen.setdefault(op.name, now)
        self._last_seen[op.name] = now

    def report(self):
        print(f"{'stage':<12} {'items':>6} {'window (s)':>11} {'items/s':>9}")
        for name, count in self.items.items():
            window = self._last_seen[name] - self._first_seen[name]
            rate = count / window if window > 0 else float("inf")
            print(f"{name:<12} {count:>6} {window:>11.3f} {rate:>9.1f}")

In [4]:
def slow_double(x):
    time.sleep(0.05)  # stand-in for an embedding call, an LLM request, ...
    return x * 2


timer = TimingObserver()
pipeline = Pipeline(range(5)).map(slow_double).filter(lambda x: x % 4 == 0)

result = list(LocalInterpreter(observers=[timer]).evaluate(pipeline))
print("result:", result)
timer.report()

result: [0, 4, 8]
stage         items  window (s)   items/s
SourceOp          5       0.218      23.0
Map               5       0.216      23.2
Filter            3       0.216      13.9


## 3. Observing failures

With a plain `map`, an exception aborts the stream. `on_error` fires with the stage that
raised — the interpreter attributes the failure to the operator whose stream raised it.

In [5]:
class ErrorObserver(StageObserver):
    def __init__(self):
        self.errors = []

    def on_error(self, op, error):
        self.errors.append((op.name, error))


def parse(x):
    if x == 3:
        raise ValueError(f"cannot parse {x}")
    return x * 100


errors = ErrorObserver()
try:
    list(LocalInterpreter(observers=[errors]).evaluate(Pipeline(range(5)).map(parse)))
except ValueError as e:
    print("pipeline aborted:", e)

print("observed:", [(stage, str(err)) for stage, err in errors.errors])

pipeline aborted: cannot parse 3
observed: [('Map', 'cannot parse 3')]


For **partial-failure** pipelines, use `map_safe`: exceptions become `Err` items that keep
flowing downstream, so observers see them as ordinary items in `before`/`after` — including
the exact input that failed, which `on_error` cannot provide.

In [6]:
from neo4j_graphrag.pipeline import Err, Ok


class ResultCountingObserver(StageObserver):
    def __init__(self):
        self.ok = 0
        self.err = 0
        self.err_stages = []

    def before(self, op, item):
        if isinstance(item, Ok):
            self.ok += 1
        elif isinstance(item, Err):
            self.err += 1
            self.err_stages.append(op.name)


counter = ResultCountingObserver()
results = list(
    LocalInterpreter(observers=[counter]).evaluate(Pipeline(range(5)).map_safe(parse))
)

print([r.value if isinstance(r, Ok) else f"Err({r.exception})" for r in results])
print(f"Ok items observed: {counter.ok}, Err items observed: {counter.err}")
print("Err seen at stages:", counter.err_stages)

[0, 100, 200, 'Err(cannot parse 3)', 400]
Ok items observed: 4, Err items observed: 1
Err seen at stages: ['TryMap']


## 4. Async stages

Observers work with `map_async_chunked` too. Items leave the async stage in bursts of
`map_batch_size` (each chunk completes together on its own `asyncio.run`), so the whole
stream finishes in ~`n / map_batch_size` × the per-item latency rather than `n` × —
visible below as all 6 items arriving within a ~0.05 s window.

Because each chunk is dispatched with `asyncio.run()`, async stages cannot be evaluated
inside an already-running event loop — such as Jupyter's. Wrap the evaluation in
`asyncio.to_thread`, as below.

In [7]:
import asyncio


async def fetch(x):
    await asyncio.sleep(0.05)
    return x * 10


timer = TimingObserver()
pipeline = Pipeline(range(6)).map_async_chunked(fetch, map_batch_size=3)

# LocalInterpreter runs each chunk in its own asyncio.run(), which cannot be
# nested inside Jupyter's running event loop — so evaluate in a thread.
result = await asyncio.to_thread(
    lambda: list(LocalInterpreter(observers=[timer]).evaluate(pipeline))
)
print("result:", result)
timer.report()

result: [0, 10, 20, 30, 40, 50]
stage         items  window (s)   items/s
SourceOp          6       0.052     115.7
MapAsyncChunked      6       0.052     116.2


## Where next

- Pass several observers at once — `LocalInterpreter(observers=[LoggingStageObserver(), timer])` — they wrap in order.
- A `StageObserver` is the natural integration point for real telemetry: emit OpenTelemetry spans per item in `before`/`after`, or ship `on_error` events to your alerting.
- Because observation happens at the interpreter level, the same pipeline definition runs observed in production and unobserved (zero overhead) in tests.